# YouTube Video Question Answering using LLM

Final Project

This notebook implements a Question Answering system that allows users to ask questions about the content of YouTube videos.

The system retrieves the video transcript, converts it into embeddings, stores them in a vector database, and allows a Large Language Model (LLM) to answer questions based on the video content.

Author: Willard Soriano | Dylan Hope Salazar  
Course: AI3  
Date: March 15, 2026

## Project Overview

Watching long YouTube videos to find specific information can be time consuming.  
This project builds a system that allows users to ask questions about the content of a YouTube video and receive answers instantly.

The system works by:

1. Extracting the transcript of a YouTube video
2. Splitting the transcript into smaller chunks
3. Converting text chunks into embeddings
4. Storing embeddings in a vector database
5. Retrieving relevant chunks when a user asks a question
6. Using a Large Language Model (LLM) to generate an answer

This approach is known as **Retrieval Augmented Generation (RAG)**.

## Tools and Technologies Used

This project uses the following tools:

- **Python**
- **LangChain** – Framework for building LLM applications
- **Ollama** – Local LLM inference (used instead of paid APIs)
- **Llama3** – Language model used for answering questions
- **YouTube Transcript API** – Extract video transcripts
- **FAISS** – Vector database for storing embeddings
- **Jupyter Notebook** – Development environment

## Using Ollama Instead of Paid APIs

Many tutorials use OpenAI API which requires an API key and usage credits.

In this project, we use **Ollama**, which allows running Large Language Models locally on a computer. This eliminates the need for API keys and allows completely free experimentation.

The Llama3 model is used to process queries and generate answers based on the retrieved transcript segments.

## System Workflow

The pipeline used in this project follows these steps:

1. Input a YouTube video URL
2. Retrieve the video transcript
3. Split transcript into smaller text chunks
4. Convert text chunks into embeddings
5. Store embeddings in a vector database (FAISS)
6. Accept user questions
7. Retrieve the most relevant transcript segments
8. Use the LLM to generate an answer

This architecture enables the model to answer questions based only on the video content.

## YouTube Videos Used

The following videos were used for testing the system:

1. Video 1: [Paste YouTube Link Here]
2. Video 2: [Paste YouTube Link Here]
3. Video 3: [Paste YouTube Link Here]

The transcripts of these videos will be processed and stored in the vector database so that the model can answer questions about them.

## Example Queries

The following queries were used to test the system:

1. Query 1: What is the main topic discussed in the video?
2. Query 2: What key points are explained by the speaker?
3. Query 3: What conclusion or summary is given in the video?

The answers generated by the model are based entirely on the transcript of the selected YouTube videos.

## Installing Required Libraries

The following cell installs all necessary libraries needed to run the project.

In [ ]:
# Install required libraries for the project

!pip install -U langchain
!pip install -U langchain-core
!pip install -U langchain-community
!pip install -U langchain-text-splitters
!pip install -U langchain-ollama

!pip install -U faiss-cpu
!pip install -U youtube-transcript-api
!pip install -U pytube

In [ ]:
import langchain
import langchain_core
import langchain_community
import langchain_classic

print("LangChain:", langchain.__version__)
print("Core:", langchain_core.__version__)
print("Community:", langchain_community.__version__)
print("Classic:", langchain_classic.__version__)

LangChain: 1.2.12
Core: 1.2.19
Community: 0.4.1
Classic: 1.0.3


## Importing Required Libraries

This section imports all Python libraries required for building the YouTube Question Answering system.

In [6]:
# Import libraries

from langchain_community.document_loaders import YoutubeLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

## Loading the Language Model

The Ollama Llama3 model will be used as the language model to generate answers from the retrieved transcript segments.

In [10]:
# Initialize the Ollama LLM

llm = ChatOllama(
    model="llama3",
    temperature=0
)

print("LLM Loaded Successfully")

LLM Loaded Successfully


## Loading YouTube Video Transcripts

In this step, the system retrieves transcripts from multiple YouTube videos using LangChain's `YoutubeLoader`.

Each video URL is provided to the loader, which automatically extracts the transcript text. These transcripts are then stored as documents that will later be split into smaller chunks for processing.

The project uses three videos related to machine learning and neural networks to demonstrate how the system can answer questions based on video content.

In [12]:
# YouTube videos to analyze

video_urls = [
    "https://www.youtube.com/watch?v=aircAruvnKk",
    "https://www.youtube.com/watch?v=IHZwWFHWa-w",
    "https://www.youtube.com/watch?v=tPYj3fFJGjk"
]

documents = []

for url in video_urls:
    loader = YoutubeLoader.from_youtube_url(
        url,
        add_video_info=False   # FIX: disable metadata request
    )
    
    docs = loader.load()
    documents.extend(docs)

print("Number of transcript documents loaded:", len(documents))

Number of transcript documents loaded: 3


## YouTube Videos Used

The following YouTube videos were used to test the question-answering system. These videos contain explanations related to artificial intelligence and machine learning concepts.

1. **But what is a Neural Network? | Deep Learning, Chapter 1**  
   https://www.youtube.com/watch?v=aircAruvnKk

2. **Gradient Descent, Step-by-Step**  
   https://www.youtube.com/watch?v=IHZwWFHWa-w

3. **Neural Networks and Deep Learning Overview**  
   https://www.youtube.com/watch?v=tPYj3fFJGjk

The transcripts from these videos are automatically retrieved using the `YoutubeLoader` in LangChain.  
The transcripts are then processed and stored in the vector database so that the model can answer questions about their content.

## Splitting the Transcript

Long transcripts must be divided into smaller chunks so that they can be processed by the language model efficiently.

In [17]:
# Split transcripts into chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(documents)

print("Total transcript chunks:", len(splits))

Total transcript chunks: 639


## Generating Embeddings and Creating the Vector Store

In this step, the transcript chunks are converted into vector embeddings using the **Ollama embedding model (`nomic-embed-text`)**.

Embeddings transform text into numerical vector representations that capture semantic meaning. This allows similar pieces of text to be identified through vector similarity.

The generated embeddings are then stored in a **FAISS vector database**, which enables efficient similarity search.

A **retriever** is created from the vector store so that relevant transcript chunks can be retrieved when a user asks a question.

In [14]:
# Create embeddings using Ollama

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

# Create vector database

vectorstore = FAISS.from_documents(
    splits,
    embeddings
)

retriever = vectorstore.as_retriever()

print("Vector store created successfully")

Vector store created successfully


## Building the Retrieval-Augmented Generation (RAG) Pipeline

To answer questions about the YouTube videos, a Retrieval-Augmented Generation (RAG) pipeline is constructed.

The pipeline performs the following steps:

1. Receive a user question
2. Retrieve the most relevant transcript chunks from the vector database
3. Format the retrieved documents into context
4. Pass the context and the question to the language model
5. Generate a final answer based on the video content

This implementation uses the modern LangChain Expression Language (LCEL) pipeline instead of older chain-based approaches.

In [18]:
# Helper function to format retrieved documents

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [19]:
prompt = ChatPromptTemplate.from_template(
"""
Answer the question based only on the context provided.

<context>
{context}
</context>

Question: {question}
"""
)

In [20]:
rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready")

RAG pipeline ready


## Example Queries Used for Testing

To evaluate the system, several questions were asked about the indexed YouTube videos.  
The model retrieves relevant transcript segments and generates answers based on the retrieved context.

The following queries were used in this project:

1. **What is the main topic discussed in the videos?**  
2. **What key ideas are explained in the videos?**  
3. **What conclusions or summaries are given?**

These queries demonstrate how the system can extract meaningful information from long video transcripts.

In [21]:
# Query 1

query = "What is the main topic discussed in the videos?"

answer = rag_chain.invoke(query)

print("Question:", query)
print("\nAnswer:")
print(answer)

Question: What is the main topic discussed in the videos?

Answer:
Based on the context, the main topic discussed in the videos is Machine Learning and Artificial Intelligence, specifically TensorFlow.


In [22]:
# Query 2

query = "What key ideas are explained in the videos?"

answer = rag_chain.invoke(query)

print("Question:", query)
print("\nAnswer:")
print(answer)

Question: What key ideas are explained in the videos?

Answer:
Based on the context, the key ideas explained in the videos are:

1. The purpose of the videos is to introduce the syntax and show how to get a working prototype, rather than going into extreme details.
2. The videos aim to provide enough knowledge so that viewers can look up and figure out more important details on their own.
3. TensorFlow is a library of tools that allows users to omit complicated math operations, making it easier to perform machine learning tasks.
4. TensorFlow has two main components: graphs and sessions, which are necessary to understand how operations and math are performed.
5. The videos will not provide in-depth explanations of why certain things work, but rather provide code and brief explanations, encouraging viewers to research and look up more detailed explanations.

These key ideas are mentioned throughout the context, providing an overview of the content and approach of the videos.


In [23]:
# Query 3

query = "What conclusions or summaries are given?"

answer = rag_chain.invoke(query)

print("Question:", query)
print("\nAnswer:")
print(answer)

Question: What conclusions or summaries are given?

Answer:
Based on the provided context, the following conclusions or summaries are given:

1. The speaker is going to show a brief summary of what they did and how it worked when they trained the model on a B movie script.
2. The B movie script is not as long as the Romeo and Juliet play, but it should be okay for the model.
3. The speaker will show the results from the B movie script to help clarify any confusion.
4. The model is being used to make predictions on movie reviews.
5. The speaker will walk through how the prediction function works and then show the actual output from the model.


## Results and Conclusion

The system successfully retrieves relevant transcript segments from the indexed YouTube videos and generates answers based on the retrieved context.

When a user asks a question, the retriever searches the FAISS vector database to identify the most relevant transcript chunks. These retrieved segments are then provided to the language model, which generates a response grounded in the video content.

The results demonstrate that Retrieval-Augmented Generation (RAG) can be effectively used to build question-answering systems for multimedia sources such as YouTube videos.

By combining transcript extraction, embeddings, vector search, and a local language model using Ollama, the system is able to analyze long video transcripts and provide concise answers to user queries.

This approach can be extended to larger collections of videos, enabling applications such as educational assistants, video summarization tools, and searchable video knowledge bases.